## SRP026279

**paper:** [PMID: 24908250](https://pubmed.ncbi.nlm.nih.gov/24908250/) - Quantitative genome-wide enhancer activity maps for five Drosophila species show functional enhancer conservation and turnover during cis-regulatory evolution, 2014

**date, curator:** 2026-06-23, Sara Carsanaro

**notes**
* rejected all STARR-seq samples which target DNA enhancers
* per methods RNA selection is polyA
* samples are adult females
* d. mel strain is Oregon-R

### annotation summary

In [23]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,follicle cells,CL:0000477,follicle cell of egg chamber,perfect match
1,follicle cells,FBbt:00004904,follicle cell,perfect match


In [24]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,adult,UBERON:0000066,fully formed stage,perfect match
1,adult,FBdv:00005369,adult stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP026279"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
0it [00:00, ?it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX467359,SRP026279,Illumina HiSeq 2000,SRS555024,CL:0000477,follicle cell of egg chamber,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1323914,follicle cells,adult,perfect match,not documented,perfect match,F,,,7245,,,,,,mRNA_Dyak_follicle_cells,"SAMN02630815,GSM1323914",,,,,,,,,,,23/06/2026,"RNA was isolated using QIAShredder (QIAGEN; cat.no. 79654) and RNeasy Mini Kit (QIAGEN; cat.no. 74106; including DNase I on column digestion) from 5-10 x 10^6 cells. 5ug total RNA was then processed as described before (S. Zhong et al., High-throughput illumina strand-specific RNA sequencing library preparation., Cold Spring Harbor protocols 2011, 940-9 (2011)) with minor adjustments. End-repair, dA-tailing and adapter ligation were performed with the NEBNext DNA Library Prep Reagent Set for Illumina (NEB, cat.no. E6000L) with proportionally reduced enzyme amounts to account for low RNA concentrations. Indexed library amplification was carried out using KAPA Library Amp Real Time (KAPA biosystems: cat.no. KK2701).",,,,follicle cells,,,,TRANSCRIPTOMIC,cDNA
1,SRX467358,SRP026279,Illumina HiSeq 2000,SRS555023,FBbt:00004904,follicle cell,FBdv:00005369,adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1323913,follicle cells,adult,perfect match,not documented,perfect match,F,Oregon-R,,7227,,,,,,mRNA_Dmel_follicle_cells,"SAMN02630817,GSM1323913",,,,,,,,,,,23/06/2026,"RNA was isolated using QIAShredder (QIAGEN; cat.no. 79654) and RNeasy Mini Kit (QIAGEN; cat.no. 74106; including DNase I on column digestion) from 5-10 x 10^6 cells. 5ug total RNA was then processed as described before (S. Zhong et al., High-throughput illumina strand-specific RNA sequencing library preparation., Cold Spring Harbor protocols 2011, 940-9 (2011)) with minor adjustments. End-repair, dA-tailing and adapter ligation were performed with the NEBNext DNA Library Prep Reagent Set for Illumina (NEB, cat.no. E6000L) with proportionally reduced enzyme amounts to account for low RNA concentrations. Indexed library amplification was carried out using KAPA Library Amp Real Time (KAPA biosystems: cat.no. KK2701).",,,,follicle cells,,,,TRANSCRIPTOMIC,cDNA


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['follicle cells']


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [6]:
unique_sorted(library, "infoStage")

['adult']


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [7]:
library.loc[:,'sex'] = 'F'
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX467359,SRP026279,Illumina HiSeq 2000,SRS555024,CL:0000477,follicle cell of egg chamber,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1323914,follicle cells,adult,perfect match,not documented,perfect match,F,,,7245,,,,,,mRNA_Dyak_follicle_cells,"SAMN02630815,GSM1323914",,,,,,,,,,,23/06/2026,"RNA was isolated using QIAShredder (QIAGEN; cat.no. 79654) and RNeasy Mini Kit (QIAGEN; cat.no. 74106; including DNase I on column digestion) from 5-10 x 10^6 cells. 5ug total RNA was then processed as described before (S. Zhong et al., High-throughput illumina strand-specific RNA sequencing library preparation., Cold Spring Harbor protocols 2011, 940-9 (2011)) with minor adjustments. End-repair, dA-tailing and adapter ligation were performed with the NEBNext DNA Library Prep Reagent Set for Illumina (NEB, cat.no. E6000L) with proportionally reduced enzyme amounts to account for low RNA concentrations. Indexed library amplification was carried out using KAPA Library Amp Real Time (KAPA biosystems: cat.no. KK2701).",,,,follicle cells,,,,TRANSCRIPTOMIC,cDNA
1,SRX467358,SRP026279,Illumina HiSeq 2000,SRS555023,FBbt:00004904,follicle cell,FBdv:00005369,adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1323913,follicle cells,adult,perfect match,not documented,perfect match,F,Oregon-R,,7227,,,,,,mRNA_Dmel_follicle_cells,"SAMN02630817,GSM1323913",,,,,,,,,,,23/06/2026,"RNA was isolated using QIAShredder (QIAGEN; cat.no. 79654) and RNeasy Mini Kit (QIAGEN; cat.no. 74106; including DNase I on column digestion) from 5-10 x 10^6 cells. 5ug total RNA was then processed as described before (S. Zhong et al., High-throughput illumina strand-specific RNA sequencing library preparation., Cold Spring Harbor protocols 2011, 940-9 (2011)) with minor adjustments. End-repair, dA-tailing and adapter ligation were performed with the NEBNext DNA Library Prep Reagent Set for Illumina (NEB, cat.no. E6000L) with proportionally reduced enzyme amounts to account for low RNA concentrations. Indexed library amplification was carried out using KAPA Library Amp Real Time (KAPA biosystems: cat.no. KK2701).",,,,follicle cells,,,,TRANSCRIPTOMIC,cDNA


#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
#my_protocol = ''
# full_length or 3'
#my_protocolType = ''

#library.loc[:,'protocol'] = my_protocol
#library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX467359,SRP026279,Illumina HiSeq 2000,SRS555024,CL:0000477,follicle cell of egg chamber,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1323914,follicle cells,adult,perfect match,not documented,perfect match,F,,,7245,,,polyA,,,mRNA_Dyak_follicle_cells,"SAMN02630815,GSM1323914",,,,,,,,,,,23/06/2026,"RNA was isolated using QIAShredder (QIAGEN; cat.no. 79654) and RNeasy Mini Kit (QIAGEN; cat.no. 74106; including DNase I on column digestion) from 5-10 x 10^6 cells. 5ug total RNA was then processed as described before (S. Zhong et al., High-throughput illumina strand-specific RNA sequencing library preparation., Cold Spring Harbor protocols 2011, 940-9 (2011)) with minor adjustments. End-repair, dA-tailing and adapter ligation were performed with the NEBNext DNA Library Prep Reagent Set for Illumina (NEB, cat.no. E6000L) with proportionally reduced enzyme amounts to account for low RNA concentrations. Indexed library amplification was carried out using KAPA Library Amp Real Time (KAPA biosystems: cat.no. KK2701).",,,,follicle cells,,,,TRANSCRIPTOMIC,cDNA
1,SRX467358,SRP026279,Illumina HiSeq 2000,SRS555023,FBbt:00004904,follicle cell,FBdv:00005369,adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1323913,follicle cells,adult,perfect match,not documented,perfect match,F,Oregon-R,,7227,,,polyA,,,mRNA_Dmel_follicle_cells,"SAMN02630817,GSM1323913",,,,,,,,,,,23/06/2026,"RNA was isolated using QIAShredder (QIAGEN; cat.no. 79654) and RNeasy Mini Kit (QIAGEN; cat.no. 74106; including DNase I on column digestion) from 5-10 x 10^6 cells. 5ug total RNA was then processed as described before (S. Zhong et al., High-throughput illumina strand-specific RNA sequencing library preparation., Cold Spring Harbor protocols 2011, 940-9 (2011)) with minor adjustments. End-repair, dA-tailing and adapter ligation were performed with the NEBNext DNA Library Prep Reagent Set for Illumina (NEB, cat.no. E6000L) with proportionally reduced enzyme amounts to account for low RNA concentrations. Indexed library amplification was carried out using KAPA Library Amp Real Time (KAPA biosystems: cat.no. KK2701).",,,,follicle cells,,,,TRANSCRIPTOMIC,cDNA


#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [11]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-06-23'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX467359,SRP026279,Illumina HiSeq 2000,SRS555024,CL:0000477,follicle cell of egg chamber,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1323914,follicle cells,adult,perfect match,not documented,perfect match,F,,,7245,,,polyA,,,mRNA_Dyak_follicle_cells,"SAMN02630815,GSM1323914",,,,,,,,,,SAC,2026-06-23,"RNA was isolated using QIAShredder (QIAGEN; cat.no. 79654) and RNeasy Mini Kit (QIAGEN; cat.no. 74106; including DNase I on column digestion) from 5-10 x 10^6 cells. 5ug total RNA was then processed as described before (S. Zhong et al., High-throughput illumina strand-specific RNA sequencing library preparation., Cold Spring Harbor protocols 2011, 940-9 (2011)) with minor adjustments. End-repair, dA-tailing and adapter ligation were performed with the NEBNext DNA Library Prep Reagent Set for Illumina (NEB, cat.no. E6000L) with proportionally reduced enzyme amounts to account for low RNA concentrations. Indexed library amplification was carried out using KAPA Library Amp Real Time (KAPA biosystems: cat.no. KK2701).",,,,follicle cells,,,,TRANSCRIPTOMIC,cDNA
1,SRX467358,SRP026279,Illumina HiSeq 2000,SRS555023,FBbt:00004904,follicle cell,FBdv:00005369,adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1323913,follicle cells,adult,perfect match,not documented,perfect match,F,Oregon-R,,7227,,,polyA,,,mRNA_Dmel_follicle_cells,"SAMN02630817,GSM1323913",,,,,,,,,,SAC,2026-06-23,"RNA was isolated using QIAShredder (QIAGEN; cat.no. 79654) and RNeasy Mini Kit (QIAGEN; cat.no. 74106; including DNase I on column digestion) from 5-10 x 10^6 cells. 5ug total RNA was then processed as described before (S. Zhong et al., High-throughput illumina strand-specific RNA sequencing library preparation., Cold Spring Harbor protocols 2011, 940-9 (2011)) with minor adjustments. End-repair, dA-tailing and adapter ligation were performed with the NEBNext DNA Library Prep Reagent Set for Illumina (NEB, cat.no. E6000L) with proportionally reduced enzyme amounts to account for low RNA concentrations. Indexed library amplification was carried out using KAPA Library Amp Real Time (KAPA biosystems: cat.no. KK2701).",,,,follicle cells,,,,TRANSCRIPTOMIC,cDNA


#### comments

In [12]:
library.loc[:,'comment'] = 'PMID: 24908250'

#### save complete file with correct columns

In [13]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX467359,SRP026279,Illumina HiSeq 2000,SRS555024,CL:0000477,follicle cell of egg chamber,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1323914,follicle cells,adult,perfect match,not documented,perfect match,F,,,7245,,,polyA,,,mRNA_Dyak_follicle_cells,"SAMN02630815,GSM1323914",,,,,,,PMID: 24908250,,,SAC,2026-06-23
1,SRX467358,SRP026279,Illumina HiSeq 2000,SRS555023,FBbt:00004904,follicle cell,FBdv:00005369,adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1323913,follicle cells,adult,perfect match,not documented,perfect match,F,Oregon-R,,7227,,,polyA,,,mRNA_Dmel_follicle_cells,"SAMN02630817,GSM1323913",,,,,,,PMID: 24908250,,,SAC,2026-06-23


### experiment annotations

In [14]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP026279,Quantitative genome-wide enhancer activity maps for five Drosophila species show functional enhancer conservation and turnover during cis-regulatory evolution,"Phenotypic differences between closely related species are thought to arise primarily from changes in gene expression due to mutations in cis-regulatory sequences (enhancers). However, it has remained unclear how frequently mutations alter enhancer activity or create functional enhancers de novo. Here we use STARR-seq, a recently developed quantitative enhancer assay, to determine genome-wide enhancer activity profiles for five Drosophila species in the constant trans-regulatory environment of Drosophila melanogaster S2 cells. We find that the functions of a large fraction of D. melanogaster enhancers are conserved for their orthologous sequences owing to selection and stabilizing turnover of transcription factor motifs. Moreover, hundreds of enhancers have been gained since the D. melanogaster-Drosophila yakuba split about 11 million years ago without apparent adaptive selection and can contribute to changes in gene expression in vivo. Our finding that enhancer activity is often deeply conserved and frequently gained provides functional insights into regulatory evolution. Overall design: STARR-seq was performed in S2 cells with paired-end sequencing in two replicates and respective inputs using genomic DNA from different Drosophila species. RNA-seq was performed in a non-stranded manner without replicates for two Drosophila species.",SRA,,,,,,GSE48251,PRJNA209379,24908250,,10.1038/ng.3009,,


#### experiment and protocol details

In [15]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

2

In [16]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
#experiment.loc[:,'protocol'] = my_protocol
#experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP026279,Quantitative genome-wide enhancer activity maps for five Drosophila species show functional enhancer conservation and turnover during cis-regulatory evolution,"Phenotypic differences between closely related species are thought to arise primarily from changes in gene expression due to mutations in cis-regulatory sequences (enhancers). However, it has remained unclear how frequently mutations alter enhancer activity or create functional enhancers de novo. Here we use STARR-seq, a recently developed quantitative enhancer assay, to determine genome-wide enhancer activity profiles for five Drosophila species in the constant trans-regulatory environment of Drosophila melanogaster S2 cells. We find that the functions of a large fraction of D. melanogaster enhancers are conserved for their orthologous sequences owing to selection and stabilizing turnover of transcription factor motifs. Moreover, hundreds of enhancers have been gained since the D. melanogaster-Drosophila yakuba split about 11 million years ago without apparent adaptive selection and can contribute to changes in gene expression in vivo. Our finding that enhancer activity is often deeply conserved and frequently gained provides functional insights into regulatory evolution. Overall design: STARR-seq was performed in S2 cells with paired-end sequencing in two replicates and respective inputs using genomic DNA from different Drosophila species. RNA-seq was performed in a non-stranded manner without replicates for two Drosophila species.",SRA,partial,Bgee 1K,2,,,GSE48251,PRJNA209379,24908250,,10.1038/ng.3009,,


#### paper and xrefs

In [17]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
#experiment.loc[:,'PMID'] = ''
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC4250274/'
#experiment.loc[:,'DOI'] = ''
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP026279,Quantitative genome-wide enhancer activity maps for five Drosophila species show functional enhancer conservation and turnover during cis-regulatory evolution,"Phenotypic differences between closely related species are thought to arise primarily from changes in gene expression due to mutations in cis-regulatory sequences (enhancers). However, it has remained unclear how frequently mutations alter enhancer activity or create functional enhancers de novo. Here we use STARR-seq, a recently developed quantitative enhancer assay, to determine genome-wide enhancer activity profiles for five Drosophila species in the constant trans-regulatory environment of Drosophila melanogaster S2 cells. We find that the functions of a large fraction of D. melanogaster enhancers are conserved for their orthologous sequences owing to selection and stabilizing turnover of transcription factor motifs. Moreover, hundreds of enhancers have been gained since the D. melanogaster-Drosophila yakuba split about 11 million years ago without apparent adaptive selection and can contribute to changes in gene expression in vivo. Our finding that enhancer activity is often deeply conserved and frequently gained provides functional insights into regulatory evolution. Overall design: STARR-seq was performed in S2 cells with paired-end sequencing in two replicates and respective inputs using genomic DNA from different Drosophila species. RNA-seq was performed in a non-stranded manner without replicates for two Drosophila species.",SRA,partial,Bgee 1K,2,,,GSE48251,PRJNA209379,24908250,https://pmc.ncbi.nlm.nih.gov/articles/PMC4250274/,10.1038/ng.3009,,


#### comments

In [18]:
experiment.loc[:,'comment'] = 'removed WGS and STARR-seq libraries'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP026279,Quantitative genome-wide enhancer activity maps for five Drosophila species show functional enhancer conservation and turnover during cis-regulatory evolution,"Phenotypic differences between closely related species are thought to arise primarily from changes in gene expression due to mutations in cis-regulatory sequences (enhancers). However, it has remained unclear how frequently mutations alter enhancer activity or create functional enhancers de novo. Here we use STARR-seq, a recently developed quantitative enhancer assay, to determine genome-wide enhancer activity profiles for five Drosophila species in the constant trans-regulatory environment of Drosophila melanogaster S2 cells. We find that the functions of a large fraction of D. melanogaster enhancers are conserved for their orthologous sequences owing to selection and stabilizing turnover of transcription factor motifs. Moreover, hundreds of enhancers have been gained since the D. melanogaster-Drosophila yakuba split about 11 million years ago without apparent adaptive selection and can contribute to changes in gene expression in vivo. Our finding that enhancer activity is often deeply conserved and frequently gained provides functional insights into regulatory evolution. Overall design: STARR-seq was performed in S2 cells with paired-end sequencing in two replicates and respective inputs using genomic DNA from different Drosophila species. RNA-seq was performed in a non-stranded manner without replicates for two Drosophila species.",SRA,partial,Bgee 1K,2,,,GSE48251,PRJNA209379,24908250,https://pmc.ncbi.nlm.nih.gov/articles/PMC4250274/,10.1038/ng.3009,,removed WGS and STARR-seq libraries


#### save complete file

In [19]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [20]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [21]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [26]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [27]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
65850,SRX23161245,SRP483125,Illumina NovaSeq 6000,SRS20111404,UBERON:0001897,dorsal plus ventral thalamus,UBERON:0034919,juvenile stage,,Thalamus,5-6 years,perfect match,not documented,other,M,,,9555,TruSeq Stranded mRNA,full_length,polyA,,,Model organism or animal sample from Papio anubis,SAMN39402824,5-6,year,,,,,,,,ANN,2026-06-23
65851,SRX23161244,SRP483125,Illumina NovaSeq 6000,SRS20111403,UBERON:0002106,spleen,UBERON:0034919,juvenile stage,,Spleen,5-6 years,perfect match,not documented,other,M,,,9555,TruSeq Stranded mRNA,full_length,polyA,,,Model organism or animal sample from Papio anubis,SAMN39402823,5-6,year,,,,,,,,ANN,2026-06-23
65852,SRX467359,SRP026279,Illumina HiSeq 2000,SRS555024,CL:0000477,follicle cell of egg chamber,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,follicle cells,adult,perfect match,not documented,perfect match,F,,,7245,,,polyA,,,mRNA_Dyak_follicle_cells,"SAMN02630815,GSM1323914",,,,,,,PMID: 24908250,,,SAC,2026-06-23
65853,SRX467358,SRP026279,Illumina HiSeq 2000,SRS555023,FBbt:00004904,follicle cell,FBdv:00005369,adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,follicle cells,adult,perfect match,not documented,perfect match,F,Oregon-R,,7227,,,polyA,,,mRNA_Dmel_follicle_cells,"SAMN02630817,GSM1323913",,,,,,,PMID: 24908250,,,SAC,2026-06-23


In [28]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1265,SRP391158,Effect of maternal nutrient reduction (MNR) du...,The objective was to determine changes in gene...,SRA,total,,56,mRNA-seq Lib Prep Kit for Illumina,full_length,GSE211085,PRJNA868787,36924159,https://pmc.ncbi.nlm.nih.gov/articles/PMC10202...,10.1017/S204017442300003X,,
1266,SRP483125,Allele specific expression analysis of 12 oliv...,Twelve age-matched male baboons were genome se...,SRA,partial,,125,TruSeq Stranded mRNA,full_length,,PRJNA1063641,40187355,https://pmc.ncbi.nlm.nih.gov/articles/PMC12143...,10.1016/j.xgen.2025.100823,,
1267,SRP026279,Quantitative genome-wide enhancer activity map...,Phenotypic differences between closely related...,SRA,partial,Bgee 1K,2,,,GSE48251,PRJNA209379,24908250,https://pmc.ncbi.nlm.nih.gov/articles/PMC4250274/,10.1038/ng.3009,,removed WGS and STARR-seq libraries


### add annotations to git

In [29]:
! git pull

Already up to date.


In [30]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [31]:
! git add $git_experiment_path $git_library_path

In [32]:
! git commit -m $commit_message_exp

[develop 1437466] adding annotated bulk experiment SRP026279
 2 files changed, 3 insertions(+)


In [33]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.65 KiB | 679.00 KiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   e917051..1437466  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push